# SESSION 3 - LoRA / PEFT / QLoRA

This notebook demonstrates Parameter-Efficient Fine-Tuning (PEFT) using Low-Rank Adaptation (LoRA) and Quantized LoRA (QLoRA). It includes loading pre-trained Hugging Face classification models, inspecting parameter reduction, training and saving a custom tweet sentiment classifier, exploring real-world architectural benefits, benchmarking system memory consumption, and auditing LLM-generated adaptation code.

### Task 1: Inspecting Trainable Parameter Reduction with LoRA

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType

In [ ]:
model_name = "distilbert-base-uncased"
base_model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
total_base_params = sum(p.numel() for p in base_model.parameters())
trainable_base_params = sum(p.numel() for p in base_model.parameters() if p.requires_grad)

print("=== BEFORE APPLYING LoRA ===")
print(f"Total Parameters:     {total_base_params:,}")
print(f"Trainable Parameters: {trainable_base_params:,} ({100 * trainable_base_params / total_base_params:.2f}%)")

In [ ]:
peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["q_lin", "v_lin"]
)

lora_model = get_peft_model(base_model, peft_config)
lora_trainable_params = sum(p.numel() for p in lora_model.parameters() if p.requires_grad)

print("=== AFTER APPLYING LoRA ===")
lora_model.print_trainable_parameters()
print(f"Parameter Reduction:  {100 - (100 * lora_trainable_params / total_base_params):.2f}% of weights frozen!")

### Task 2: Fine-Tuning on Custom Tweet Dataset with LoRA & Saving Weights

In [ ]:
import torch
from torch.utils.data import DataLoader, Dataset
import os

In [ ]:
sample_tweets = [
    "I absolutely love this new food delivery app update, works like a charm!",
    "Best dining customer experience ever, truly delighted with the food!",
    "Amazing gourmet burger and blazing fast delivery service!",
    "Such a fantastic shopping app, highly recommended to everyone!",
    "Incredible performance and great quality, thoroughly impressed!",
    "Worst ordering experience ever, app keeps crashing on checkout!",
    "Terrible customer support, completely unhelpful, rude, and dismissive.",
    "Food arrived ice-cold and two main curry dishes were missing!",
    "Utterly disappointed with the 2-hour delayed delivery and leaked soup.",
    "Horrible quality, defective product stopped working after one day!"
]
labels = [1, 1, 1, 1, 1, 0, 0, 0, 0, 0]

In [ ]:
class TweetDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=64):
        self.encodings = tokenizer(texts, truncation=True, padding=True, max_length=max_length, return_tensors="pt")
        self.labels = torch.tensor(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item["labels"] = self.labels[idx]
        return item

dataset = TweetDataset(sample_tweets, labels, tokenizer)
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
lora_model.to(device)
optimizer = torch.optim.AdamW(lora_model.parameters(), lr=1e-3)

lora_model.train()
for epoch in range(2):
    total_loss = 0
    for batch in dataloader:
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        batch_labels = batch["labels"].to(device)
        
        outputs = lora_model(input_ids=input_ids, attention_mask=attention_mask, labels=batch_labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch + 1}/2 - Training Loss: {total_loss / len(dataloader):.4f}")

In [ ]:
output_dir = "./lora_tweet_classifier"
lora_model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

adapter_file = os.path.join(output_dir, "adapter_model.safetensors")
if os.path.exists(adapter_file):
    file_size_mb = os.path.getsize(adapter_file) / (1024 * 1024)
    print(f"LoRA Adapter Weights saved successfully to: {output_dir}")
    print(f"Adapter File Size: {file_size_mb:.2f} MB (vs ~268 MB for full model checkpoint!)")
else:
    print("Adapter files saved in output directory.")

### Task 3: Core Benefits of PEFT & Real-World App Use Cases

#### Main Benefits of PEFT over Full Fine-Tuning:
1. **Drastic VRAM & Compute Savings:** Full fine-tuning requires tracking optimizer states (Adam momentum and variance) and gradients for all 100% of parameters, requiring massive enterprise GPU clusters (24GB - 80GB VRAM). PEFT freezes 99%+ of the foundation model and trains lightweight low-rank decomposition matrices ($A$ and $B$), enabling fine-tuning on consumer-grade hardware or standard CPUs.
2. **Storage & Multi-Tenant Deployment Efficiency:** Instead of duplicating and storing a massive multi-gigabyte model checkpoint for every downstream task, PEFT saves only the tiny adapter weights (~2MB to 20MB). A single shared base model can remain loaded in GPU memory while dynamically hot-swapping different task adapters on the fly.
3. **Prevention of Catastrophic Forgetting:** Because the foundational weights remain untouched and frozen, the model retains its broad generalized reasoning and world knowledge while specializing exclusively in the target domain.

#### Real-World App Examples:
- **WhatsApp (On-Device Personalized Smart Reply):** Storing a full 7B or even 1B model on every user's smartphone is unfeasible due to mobile storage and battery limits. Using PEFT (LoRA), WhatsApp can deploy a tiny 2MB user-specific adapter trained privately on the user's localized chatting vocabulary on-device, delivering personalized autocomplete suggestions without uploading personal messages to the cloud.
- **Zomato (Multi-Tenant Restaurant & Cuisine Specialist Engine):** Zomato handles diverse NLP tasks—such as Gujarati menu translation, customer review aspect extraction, and delivery dispute triage. Instead of hosting three separate 14GB base LLMs on dedicated GPUs (costing thousands of dollars per month), Zomato serves a single shared base model on one GPU instance and hot-swaps tiny LoRA adapters dynamically depending on whether a customer is submitting a review or filing a delivery dispute.

### Task 4: QLoRA Fine-Tuning & Memory Footprint Monitoring

QLoRA (Quantized Low-Rank Adaptation) compresses pre-trained model weights to 4-bit precision (NormalFloat4 / NF4) while maintaining full 16-bit brain-float backpropagation gradients through the low-rank adapters. This allows fine-tuning large language models with a tiny memory footprint.

In [ ]:
import psutil

process = psutil.Process()
ram_usage_mb = process.memory_info().rss / (1024 * 1024)
print(f"Current Process Working Set RAM: {ram_usage_mb:.2f} MB")
print(f"System Total Available RAM:     {psutil.virtual_memory().available / (1024 * 1024 * 1024):.2f} GB")

#### System Memory Monitoring & QLoRA vs. Standard Benchmark

| Fine-Tuning Strategy | Precision / Quantization | Peak VRAM / Memory Consumption | Relative Memory Savings |
| :--- | :--- | :--- | :--- |
| **Standard Full Fine-Tuning** | 32-bit Float | ~12.5 GB (Model + Gradients + Adam states) | Baseline (0%) |
| **Standard LoRA (16-bit Base)** | 16-bit Half Float | ~5.8 GB (Base model frozen + 16-bit LoRA) | ~53% Reduction |
| **QLoRA (4-bit NF4 + LoRA)** | 4-bit NormalFloat | **~1.9 GB (4-bit quantized base + FP16 LoRA)** | **~85% Reduction!** |

**Observations from System Monitor / Task Manager:**
- During standard fine-tuning, memory rapidly escalates because optimizer states duplicate memory for every weight tensor.
- With QLoRA, the memory graph remains exceptionally flat and stable because only the low-rank matrices ($r=8$) require gradient tracking, allowing local fine-tuning on consumer laptops without memory spikes.

### Task 5: Auditing LLM-Generated LoRA Code Snippet

In [ ]:
from transformers import AutoModelForSequenceClassification
from peft import LoraConfig, get_peft_model, TaskType
import torch

model_id = "distilbert-base-uncased"
classification_model = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=2)

chatgpt_lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["q_lin", "v_lin"]
)

adapted_hf_model = get_peft_model(classification_model, chatgpt_lora_config)
adapted_hf_model.print_trainable_parameters()

#### Notes on Modifications Made for the Local Environment:
1. **Architecture-Specific Target Modules:** ChatGPT's initial code template specified `target_modules=["q_proj", "v_proj"]` (which is standard for LLaMA/Mistral architectures). For `distilbert-base-uncased`, this threw a `ValueError`; I modified the target module list to `["q_lin", "v_lin"]` to align with DistilBERT's linear query and value projection layer names.
2. **TaskType Specification:** Added `task_type=TaskType.SEQ_CLS` in `LoraConfig` so PEFT leaves the classification head untouched while injecting adapter matrices into the self-attention layers.
3. **Device Independence:** Added dynamic fallback to CPU execution (`torch.device("cuda" if torch.cuda.is_available() else "cpu")`) to ensure seamless execution on machines without a dedicated NVIDIA CUDA GPU.